In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd

In [3]:
tf.__version__ , np.__version__,pd.__version__

('2.21.0', '2.3.5', '2.3.3')

In [5]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [6]:
# CONFIG
DATA_DIR = r"D:\Proper_Dataset" # <-- change this
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
CHECKPOINT_PATH = r"C:\Users\Dell\OneDrive\Desktop\Agri_AI\checkpoints\crop_model.weights.h5"

In [7]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode = 'int'
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode = 'int'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)

Found 59497 files belonging to 40 classes.
Using 47598 files for training.
Found 59497 files belonging to 40 classes.
Using 11899 files for validation.
Classes: ['Maize___Maize Lethal Necrosis', 'Maize___Maize Rust', 'Maize___Maize fall armyworm', 'Maize___Maize grasshoper', 'Maize___Maize healthy', 'Maize___Maize leaf beetle', 'Maize___Maize leaf blight', 'Maize___Maize leaf spot', 'Maize___Maize streak virus', 'Rice___Bacterial Leaf Blight', 'Rice___Brown Spot', 'Rice___Healthy Rice Leaf', 'Rice___Leaf Blast', 'Rice___Leaf scald', 'Rice___Narrow Brown Leaf Spot', 'Rice___Rice Hispa', 'Rice___Sheath Blight', 'Sugarcane___Banded Chlorosis', 'Sugarcane___Brown Spot', 'Sugarcane___BrownRust', 'Sugarcane___Dried', 'Sugarcane___Grassy shoot', 'Sugarcane___Healthy', 'Sugarcane___Pokkah Boeng', 'Sugarcane___Sett Rot', 'Sugarcane___Viral Disease', 'Sugarcane___Yellow Leaf', 'Sugarcane___smut', 'Wheat___BlackPoint', 'Wheat___FusariumFootRot', 'Wheat___HealthyLeaf', 'Wheat___LeafBlight', 'Wheat

In [5]:
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Sample labels:", labels[:10])

Images shape: (32, 224, 224, 3)
Labels shape: (32,)
Sample labels: tf.Tensor([ 5  2  5  6 32  7  6 16 24 19], shape=(10,), dtype=int32)


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [7]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

In [8]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # Freeze base model initially

In [25]:
inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [26]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential_1 (Sequential)            │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ MobileNetV3Large (Functional)        │ (None, 7, 7, 960)           │       2,996,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 960)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 960)                 │           3,840 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │         246,016 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 40)                  │          10,280 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,256,488 (12.42 MB)

 Trainable params: 258,216 (1008.66 KB)

 Non-trainable params: 2,998,272 (11.44 MB)

In [9]:
checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath = r"C:\Users\Dell\OneDrive\Desktop\Agri_AI\checkpoints\crop_model_{epoch:02d}.keras",
    monitor="val_loss",
    save_best_only=False,
    save_weights_only=False,
    verbose=1
)

early_stop_cb = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

backup_cb = keras.callbacks.BackupAndRestore(
    backup_dir=r"C:\Users\Dell\OneDrive\Desktop\Agri_AI\backup"
)

In [11]:
initial_epoch = 8

if os.path.exists(CHECKPOINT_PATH + ".index"):
    print("Loading checkpoint...")
    model.load_weights(CHECKPOINT_PATH)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=[checkpoint_cb, early_stop_cb,backup_cb, reduce_lr_cb]
)

Epoch 9/30
1488/1488 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step - accuracy: 0.8552 - loss: 0.4123

In [3]:
from tensorflow import keras

model = keras.models.load_model(
    r"C:\Users\Dell\OneDrive\Desktop\Agri_AI\checkpoints\crop_model_08.keras"
)

In [9]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = r"D:\Proper_Dataset\Rice___Leaf Blast\Rice_Leaf Blast_augmented_image_16.jpg"
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

img_array = tf.keras.applications.mobilenet_v3.preprocess_input(img_array)

pred = model.predict(img_array)
pred_class = class_names[np.argmax(pred)]

print("Prediction:", pred_class)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction: Rice___Leaf Blast
